In [ ]:
'''
This module provides functions to fetch and process historical weather data for train stations.

It contains the following main functions:

- get_weather_for_station_and_datetime: 
    Retrieves historical weather data for a given station and date. Uses caching to avoid 
    repeated API calls and handles missing stations.

- enrich_duckdb_with_weather:
    Fills a DuckDB train delay table with historical weather data for each station and date. 
    The function retrieves unique station-date combinations, fetches weather data using 
    get_weather_for_station_and_datetime, periodically saves progress to a file, and writes 
    the filled table back to the database.

Author: Eduard Unruh (Student ID: 20170814)
'''

# Change working directory so that we can import the api functions

import os
os.chdir("C:/Users/unruh/Documents/2025_11_26 Python_Project/TBA_project")

import duckdb
import pandas as pd
import requests
from src.bjarne_api.collector import get_station_details, get_historical_weather

# Create a progress file to save progress 
PROGRESS_FILE = "train_delay_weather_progress.parquet"
SAVE_EVERY = 200


# create a cache so that we don't have to calculate the same values multiple times
station_cache = {}
weather_cache = {}

# Put the get_historical_weather and the get_station_details functions together
def get_weather_for_station_and_datetime(station_name, dt):
    
    date_str = dt.strftime("%Y-%m-%d")      # Weather api needs date not timestamp
    cache_key = (station_name, date_str)    # Initialise cache key

    # Check if there is already a value in the cache
    if cache_key in weather_cache:
        weather_data = weather_cache[cache_key]
    else:
        # Get station coordinates
        if station_name in station_cache:
            lat, lon = station_cache[station_name]
        else:
            result = get_station_details(station_name)
            if result is None:
                return {"temperature": None, "precipitation": None, "wind_speed": None, "weather_status": "Station not found"}
            id, lat, lon = result
            station_cache[station_name] = (lat, lon)

        # Fetch historical weather
        weather_data = get_historical_weather(lat, lon, date_str)
        weather_cache[cache_key] = weather_data

    return {
    "temperature": weather_data.get("temp_avg"),
    "precipitation": weather_data.get("precipitation_sum"),
    "wind_speed": weather_data.get("wind_speed_max"),
    "weather_status": weather_data.get("weather_status", "Error")
}


# Main pipeline
import time
from datetime import timedelta

def enrich_duckdb_with_weather(
    db_path="data/train.duckdb",
    table_name="train_delay",
    new_table_name="train_delay_with_weather"
):

    # Connect to our database
    con = duckdb.connect(db_path)

    # Get the db as a df
    pairs_df = con.execute("""
    SELECT DISTINCT
        station_current,
        CAST(event_time AS DATE) AS date -- To get the timestamp into date format
    FROM train_delay
    WHERE station_current IS NOT NULL
""").fetchdf()

    

    # Prepare weather columns
    for col in ["temperature", "precipitation", "wind_speed", "weather_status"]:
        if col not in pairs_df.columns:
            pairs_df[col] = None

    # Resume if progress file exists
    if os.path.exists(PROGRESS_FILE):
        pairs_df = pd.read_parquet(PROGRESS_FILE)
        start_idx = pairs_df["temperature"].notna().sum()
        print(f"Resuming from row {start_idx}")
    else:
        start_idx = 0

    total_rows = len(pairs_df)
    start_time = time.time()

    for i in range(start_idx, total_rows):
        row = pairs_df.iloc[i]
        weather = get_weather_for_station_and_datetime(
            row["station_current"],
            pd.Timestamp(row["date"])
        )

        # added because of filling NAs
        pairs_df.at[i, "temperature"] = weather["temperature"]
        pairs_df.at[i, "precipitation"] = weather["precipitation"]
        pairs_df.at[i, "wind_speed"] = weather["wind_speed"]
        pairs_df.at[i, "weather_status"] = weather["weather_status"]

        time.sleep(2)

        # save every x rows or at the last read row
        if (i+1) % SAVE_EVERY == 0 or i == total_rows - 1:
            pairs_df.to_parquet(PROGRESS_FILE)
            elapsed = time.time() - start_time
            done = i + 1
            rate = elapsed / done
            remaining = rate * (total_rows - done)

            print(
                f"[{done}/{total_rows} retries] "
                f"{done/total_rows:.1%} | "
                f"Elapsed: {timedelta(seconds=int(elapsed))} | "
                f"ETA: {timedelta(seconds=int(remaining))}"
            )


    # Write final result to DuckDB
    con.execute("DROP TABLE IF EXISTS station_date_weather")
    con.execute("CREATE TABLE station_date_weather AS SELECT * FROM pairs_df")

    # Join the train_delay table with the station_date_weather table on station_current
    con.execute(f"""
        DROP TABLE IF EXISTS {new_table_name};
        CREATE TABLE {new_table_name} AS
        SELECT
            t.*,
            w.temperature,
            w.precipitation,
            w.wind_speed,
            w.weather_status
        FROM {table_name} t
        LEFT JOIN station_date_weather w
        ON t.station_current = w.station_current
        AND CAST(t.event_time AS DATE) = w.date
    """)


    con.close()

    return pairs_df

In [ ]:
# Run the function to add the weather data to our database 
df_enriched = enrich_duckdb_with_weather()

In [ ]:
# Connect to db and export it into a parquet file

con = duckdb.connect("data/train.duckdb")

con.execute("""
    COPY
    (SELECT *  EXCLUDE (event_time) FROM train_delay_with_weather)
    TO 'data/train_delay_with_weather.parquet'
    (FORMAT parquet)
""")

con.close()